# Notebook Purpose

The purpose of this Jupyter notebook is to verify that the graphs created by the `torch_geometric` (PyG) package via the `pyg_with_bonds` feature type are **exactly** the same as the graphs created previously by Daniel via the DGL (& DGL Life) packages (`dgl_with_bonds` feature type), now that `dgl` has been deprecated.

For every molecule in every dataset source, the following are checked:

1. **Rows** - both processed datasets contain the same molecules, in the same order, with the same labels (all non-`mol` columns are identical).
2. **Graph size** - same number of nodes and edges.
3. **Edge ordering** - identical `(src, dst)` pairs in the same order (bond edges `u -> v`, `v -> u`, followed by self-loops).
4. **Node features** - `g.ndata['feats']` is bit-for-bit equal to `data.x` (shape `(num_atoms, 177)`).
5. **Edge features** - `g.edata['edge_feats']` is bit-for-bit equal to `data.edge_attr` (shape `(num_edges, 17)`, self-loops filled with ones).

Any molecule that differs is then classified into one of the following categories:

| Category | Meaning | Impact on replication |
|---|---|---|
| `bondless` | Molecule has no bonds (e.g. `[Hg+2]`, `[I-].[K+]` after standardisation). DGL stored the self-loop edge features with shape `(num_atoms,)` instead of `(num_atoms, 17)`, whereas PyG stores `(num_atoms, 17)` of ones. | None, **as long as** the PyG train/test strategy drops molecules without bonds, as the deprecated `DGLBondsTrainTestStrategy` did (`edge_feats.shape[-1] < 17`). |
| `rdkit_drift` | Re-running the **original** deprecated dgllife code in the current environment gives exactly the PyG graph, i.e. the code is equivalent and the difference comes from the RDKit / standardiser versions used when the DGL datasets were created. | Only these few molecules have slightly different features (e.g. explicit valence, bond stereo). |
| `unexplained` | Anything else - a real difference between the two implementations. | Must be investigated. |

Finally, a random sample of SMILES strings is re-featurised with **both** the current `PyGBondsDatasetCreator` and the original dgllife code, to check that the two implementations are equivalent in the current environment.

## Imports

Import any modules required.

**Note:** `dgl` (and `dgllife`) are only needed to unpickle the old `dgl_with_bonds` datasets and to re-run the original featurisation code. `dgl==2.1.0` does not ship a graphbolt library for `torch==2.6.0`, and fails on import. Graphbolt is only used for graph sampling (not needed here), so it is stubbed out before `dgl` is imported.

In [1]:
%pip install dgl==2.1.0 dgllife torchdata==0.9.0 pydantic


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import gc
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# make the project root importable so the `datasets` package resolves
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# stub out graphbolt so that `dgl` can be imported with torch 2.6 (see note above)
sys.modules['dgl.graphbolt'] = types.ModuleType('dgl.graphbolt')
import dgl
import rdkit
from dgllife import utils
from rdkit import RDLogger
from chembl_structure_pipeline import standardizer
from torch_geometric.data import Data

from eda.helpers import *
from datasets.dataset_handler import DatasetHandler
from datasets.enums import DatasetSource, FeatureType
from datasets.creation_strategies.pyg_with_bonds import PyGBondsDatasetCreator

# silence the (very verbose) tautomer enumeration / kekulisation logs while re-featurising
RDLogger.DisableLog("rdApp.*")

print(f"dgl: {dgl.__version__}")
print(f"torch: {torch.__version__}")
print(f"rdkit: {rdkit.__version__}")

dgl: 2.1.0
torch: 2.6.0+cpu
rdkit: 2026.09.1


## Attributes

In [3]:
handler = DatasetHandler()

dgl_with_bonds = FeatureType.DGL_WITH_BONDS
pyg_with_bonds = FeatureType.PYG_WITH_BONDS

# dataset sources
tox21_source = DatasetSource.TOX21
muv_source = DatasetSource.MUV
dude_source = DatasetSource.DUDE_GPCR

# number of molecules per dataset to re-featurise with the current `PyGBondsDatasetCreator`
REFEATURISE_SAMPLE_SIZE = 1_000
SEED = 42

# collects the results of each dataset comparison
results: dict[str, dict] = {}

## Comparison Functions

Functions used to compare a single DGL graph against a single PyG graph, and a full DGL dataset against a full PyG dataset.

In [4]:
def compare_graphs(dgl_graph: dgl.DGLGraph, pyg_graph: Data) -> list[str]:
    """
    Compares a DGL graph against a PyG graph and returns the checks that failed.

    Args:
        dgl_graph (dgl.DGLGraph): Graph created by the deprecated `dgl_with_bonds` strategy.
        pyg_graph (Data): Graph created by the `pyg_with_bonds` strategy.

    Returns:
        list[str]: Names of the failed checks (empty if the graphs are identical).
    """
    failed = []

    if dgl_graph.num_nodes() != pyg_graph.num_nodes:
        failed.append("num_nodes")
    if dgl_graph.num_edges() != pyg_graph.num_edges:
        failed.append("num_edges")

    # dgl stores edge ids as int32, pyg as int64; the values are what matter
    src, dst = dgl_graph.edges()
    dgl_edge_index = torch.stack([src, dst]).long()
    if dgl_edge_index.shape != pyg_graph.edge_index.shape or not torch.equal(dgl_edge_index, pyg_graph.edge_index):
        failed.append("edge_index")

    dgl_x = dgl_graph.ndata["feats"]
    if dgl_x.dtype != pyg_graph.x.dtype or dgl_x.shape != pyg_graph.x.shape or not torch.equal(dgl_x, pyg_graph.x):
        failed.append("node_feats")

    dgl_edge_attr = dgl_graph.edata["edge_feats"]
    if (
        dgl_edge_attr.dtype != pyg_graph.edge_attr.dtype
        or dgl_edge_attr.shape != pyg_graph.edge_attr.shape
        or not torch.equal(dgl_edge_attr, pyg_graph.edge_attr)
    ):
        failed.append("edge_feats")

    return failed


def featurise_with_dgllife(smiles: str) -> dgl.DGLGraph | None:
    """
    Featurises a SMILES string using the original (deprecated) `DGLBondsDatasetCreator`
    code, so that it can be re-run in the current environment.

    Args:
        smiles (str): A SMILES string representing a molecular structure.

    Returns:
        dgl.DGLGraph | None: The DGL graph, or None if the SMILES string could not be parsed.
    """
    # `_parse_smiles` is shared between the DGL and PyG strategies (base class)
    mol = PyGBondsDatasetCreator()._parse_smiles(smiles)
    if mol is None:
        return None

    mol = standardizer.standardize_mol(mol)

    atom_features = utils.ConcatFeaturizer([
        utils.atom_type_one_hot,
        utils.atomic_number_one_hot,
        utils.atom_degree_one_hot,
        utils.atom_explicit_valence_one_hot,
        utils.atom_formal_charge_one_hot,
        utils.atom_num_radical_electrons_one_hot,
        utils.atom_hybridization_one_hot,
        utils.atom_is_aromatic_one_hot,
    ])
    bond_features = utils.ConcatFeaturizer([
        utils.bond_type_one_hot,
        utils.bond_is_conjugated_one_hot,
        utils.bond_is_in_ring_one_hot,
        utils.bond_stereo_one_hot,
        utils.bond_direction_one_hot,
    ])

    def featurise_atoms(m) -> dict:
        return {"feats": torch.tensor([atom_features(atom) for atom in m.GetAtoms()]).float()}

    def featurise_bonds(m) -> dict:
        feats = []
        for bond in m.GetBonds():
            feats.append(bond_features(bond))
            feats.append(bond_features(bond))
        return {"edge_feats": torch.tensor(feats).float()}

    dgl_graph = utils.mol_to_bigraph(
        mol=mol,
        node_featurizer=featurise_atoms,
        edge_featurizer=featurise_bonds,
        canonical_atom_order=True
    )
    return dgl.add_self_loop(dgl_graph)


def is_bondless_difference(dgl_graph: dgl.DGLGraph, pyg_graph: Data, failed: list[str]) -> bool:
    """
    Returns True if the only difference is the known shape difference for molecules without bonds.

    For molecules without bonds, DGL ends up with self-loop edge features of shape
    `(num_atoms,)` (there is no bond feature tensor to infer the width from), whereas
    PyG creates `(num_atoms, 17)` ones. The deprecated DGL train/test strategy dropped
    these molecules (`edge_feats.shape[-1] < 17`).
    """
    num_atoms = pyg_graph.num_nodes
    return (
        failed == ["edge_feats"]
        and pyg_graph.num_edges == num_atoms  # self-loops only, i.e. no bonds
        and torch.equal(dgl_graph.edata["edge_feats"], torch.ones(num_atoms))
        and torch.equal(pyg_graph.edge_attr, torch.ones((num_atoms, 17)))
    )


def classify_difference(dgl_graph: dgl.DGLGraph, pyg_graph: Data, failed: list[str], smiles: str) -> str:
    """
    Classifies a difference between a DGL and a PyG graph (see the table at the top of the notebook).

    Returns:
        str: One of 'bondless', 'rdkit_drift' or 'unexplained'.
    """
    if is_bondless_difference(dgl_graph, pyg_graph, failed):
        return "bondless"

    # re-run the original dgllife code in the current environment; if it matches PyG,
    # the implementations are equivalent and the stored DGL graph is out of date
    rebuilt = featurise_with_dgllife(smiles)
    if rebuilt is not None:
        rebuilt_failed = compare_graphs(rebuilt, pyg_graph)
        if not rebuilt_failed or is_bondless_difference(rebuilt, pyg_graph, rebuilt_failed):
            return "rdkit_drift"

    return "unexplained"


def compare_datasets(dgl_df: pd.DataFrame, pyg_df: pd.DataFrame) -> dict:
    """
    Compares a `dgl_with_bonds` processed dataset against a `pyg_with_bonds` one.

    Args:
        dgl_df (pd.DataFrame): Processed dataset with DGL graphs in the 'mol' column.
        pyg_df (pd.DataFrame): Processed dataset with PyG graphs in the 'mol' column.

    Returns:
        dict: Summary of the comparison, including one row per differing molecule.
    """
    summary = {
        "dgl_shape": dgl_df.shape,
        "pyg_shape": pyg_df.shape,
        "same_columns": list(dgl_df.columns) == list(pyg_df.columns),
        "same_non_mol_columns": False,
        "differences": pd.DataFrame(columns=["index", "smiles", "failed_checks", "category"]),
    }

    # labels, ids and smiles must line up exactly, otherwise the graph comparison is meaningless
    try:
        pd.testing.assert_frame_equal(
            dgl_df.drop(columns="mol"),
            pyg_df.drop(columns="mol"),
            check_dtype=True,
        )
        summary["same_non_mol_columns"] = True
    except AssertionError as e:
        print(f"Non-mol columns differ:\n{e}")
        return summary

    differences = []
    for idx, (smiles, dgl_graph, pyg_graph) in enumerate(zip(dgl_df["smiles"], dgl_df["mol"], pyg_df["mol"])):
        failed = compare_graphs(dgl_graph, pyg_graph)
        if failed:
            differences.append({
                "index": idx,
                "smiles": smiles,
                "failed_checks": failed,
                "category": classify_difference(dgl_graph, pyg_graph, failed, smiles),
            })

    if differences:
        summary["differences"] = pd.DataFrame(differences)
    return summary


def refeaturise_sample(df: pd.DataFrame, n: int, seed: int) -> dict:
    """
    Re-featurises a random sample of SMILES strings with both the current `PyGBondsDatasetCreator`
    and the original dgllife code, and compares the two resulting graphs.

    Args:
        df (pd.DataFrame): Processed dataset with a 'smiles' column.
        n (int): Number of molecules to sample.
        seed (int): Random seed for the sample.

    Returns:
        dict: Number of molecules checked and one row per differing molecule.
    """
    creator = PyGBondsDatasetCreator()
    sample = df.sample(n=min(n, len(df)), random_state=seed)

    differences = []
    for idx, smiles in sample["smiles"].items():
        pyg_graph = creator._create_features(smiles)
        dgl_graph = featurise_with_dgllife(smiles)
        if pyg_graph is None or dgl_graph is None:
            if (pyg_graph is None) != (dgl_graph is None):
                differences.append({"index": idx, "smiles": smiles, "failed_checks": ["unparseable"], "category": "unexplained"})
            continue

        failed = compare_graphs(dgl_graph, pyg_graph)
        if failed:
            category = "bondless" if is_bondless_difference(dgl_graph, pyg_graph, failed) else "unexplained"
            differences.append({"index": idx, "smiles": smiles, "failed_checks": failed, "category": category})

    return {
        "checked": len(sample),
        "differences": pd.DataFrame(differences, columns=["index", "smiles", "failed_checks", "category"]),
    }


CATEGORIES = ["bondless", "rdkit_drift", "unexplained"]


def report(name: str, summary: dict, refeaturised: dict) -> None:
    """
    Prints a human-readable report of a dataset comparison.

    Args:
        name (str): Name of the dataset source.
        summary (dict): Output of `compare_datasets`.
        refeaturised (dict): Output of `refeaturise_sample`.
    """
    print(f"=== {name} ===")
    print(f"DGL shape: {summary['dgl_shape']} | PyG shape: {summary['pyg_shape']}")
    print(f"Same columns: {summary['same_columns']}")
    print(f"Same non-mol columns (ids, smiles, labels): {summary['same_non_mol_columns']}")

    stored = summary["differences"]
    total = summary["pyg_shape"][0]
    print(f"\nStored graphs: {total - len(stored):,} of {total:,} identical")
    for category in CATEGORIES:
        print(f"  {category}: {(stored['category'] == category).sum():,}")

    sampled = refeaturised["differences"]
    print(f"\nRe-featurised sample (current PyG code vs original dgllife code): {refeaturised['checked'] - len(sampled):,} of {refeaturised['checked']:,} identical")
    for category in CATEGORIES:
        print(f"  {category}: {(sampled['category'] == category).sum():,}")

    if len(stored):
        print("\nStored graph differences (first 10 per category):")
        display(stored.groupby("category").head(10))


def is_equivalent(summary: dict, refeaturised: dict) -> bool:
    """
    Returns True if a dataset comparison found no unexplained differences.
    """
    return (
        summary["dgl_shape"] == summary["pyg_shape"]
        and summary["same_columns"]
        and summary["same_non_mol_columns"]
        and not (summary["differences"]["category"] == "unexplained").any()
        and not (refeaturised["differences"]["category"] == "unexplained").any()
    )

## Tox21

Compare molecules created for the tox21 dataset.

### DGL with bonds

In [5]:
tox_dgl_df = handler.load_processed_dataset(
    dataset_source=tox21_source,
    feature_type=dgl_with_bonds
)
show_df(df=tox_dgl_df)
show_first_value(df=tox_dgl_df, column="mol")

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles,mol
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1,"Graph(num_nodes=16, num_edges=50,\n ndata..."
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O,"Graph(num_nodes=15, num_edges=47,\n ndata..."
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...,"Graph(num_nodes=21, num_edges=69,\n ndata..."
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C,"Graph(num_nodes=20, num_edges=60,\n ndata..."
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O,"Graph(num_nodes=11, num_edges=31,\n ndata..."
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,TOX5110,CC(C)(C)OOC(C)(C)CCC(C)(C)OOC(C)(C)C,"Graph(num_nodes=20, num_edges=58,\n ndata..."
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX6619,O=S(=O)(Cl)c1ccccc1,"Graph(num_nodes=10, num_edges=30,\n ndata..."
7,0.0,NaN,0.0,NaN,1.0,NaN,NaN,1.0,0.0,1.0,0.0,1.0,TOX25232,O=C(O)Cc1cc(I)c(Oc2ccc(O)c(I)c2)c(I)c1,"Graph(num_nodes=21, num_edges=65,\n ndata..."
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,TOX22514,OC[C@H](O)[C@@H](O)[C@H](O)CO,"Graph(num_nodes=10, num_edges=28,\n ndata..."
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX22517,CCCCCCCC(=O)[O-].CCCCCCCC(=O)[O-].[Zn+2],"Graph(num_nodes=10, num_edges=28,\n ndata..."


mol: Graph(num_nodes=16, num_edges=50,
      ndata_schemes={'feats': Scheme(shape=(177,), dtype=torch.float32)}
      edata_schemes={'edge_feats': Scheme(shape=(17,), dtype=torch.float32)})


In [6]:
explore_df(tox_dgl_df)

Shape: 7,823 rows x 15 columns

Info:
<class 'pandas.DataFrame'>
RangeIndex: 7823 entries, 0 to 7822
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   NR-AR          7258 non-null   float64
 1   NR-AR-LBD      6751 non-null   float64
 2   NR-AhR         6542 non-null   float64
 3   NR-Aromatase   5815 non-null   float64
 4   NR-ER          6186 non-null   float64
 5   NR-ER-LBD      6948 non-null   float64
 6   NR-PPAR-gamma  6443 non-null   float64
 7   SR-ARE         5825 non-null   float64
 8   SR-ATAD5       7065 non-null   float64
 9   SR-HSE         6460 non-null   float64
 10  SR-MMP         5804 non-null   float64
 11  SR-p53         6767 non-null   float64
 12  mol_id         7823 non-null   str    
 13  smiles         7823 non-null   str    
 14  mol            7823 non-null   object 
dtypes: float64(12), object(1), str(2)
memory usage: 916.9+ KB

Describe:


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles,mol
count,7258.000000,6751.000000,6542.000000,5815.000000,6186.000000,6948.000000,6443.000000,5825.000000,7065.000000,6460.000000,5804.000000,6767.000000,7823,7823,7823
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7823,7823,7823
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1,"Graph(num_nodes=16, num_edges=50,\n ndata..."
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1
mean,0.042436,0.035106,0.117395,0.051591,0.127869,0.050230,0.028869,0.161717,0.037367,0.057585,0.158167,0.062509,NaN,NaN,NaN
std,0.201595,0.184061,0.321915,0.221218,0.333971,0.218436,0.167450,0.368222,0.189674,0.232975,0.364929,0.242096,NaN,NaN,NaN
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN



Sample:


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles,mol
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1,"Graph(num_nodes=16, num_edges=50,\n ndata..."
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O,"Graph(num_nodes=15, num_edges=47,\n ndata..."
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...,"Graph(num_nodes=21, num_edges=69,\n ndata..."
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C,"Graph(num_nodes=20, num_edges=60,\n ndata..."
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O,"Graph(num_nodes=11, num_edges=31,\n ndata..."


### PyG with bonds

In [7]:
tox_pyg_df = handler.load_processed_dataset(
    dataset_source=tox21_source,
    feature_type=pyg_with_bonds
)
show_df(df=tox_pyg_df)
show_first_value(df=tox_pyg_df, column="mol")

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles,mol
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,TOX5110,CC(C)(C)OOC(C)(C)CCC(C)(C)OOC(C)(C)C,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX6619,O=S(=O)(Cl)c1ccccc1,"[(x, [tensor([0., 0., 1., 0., 0., 0., 0., 0., ..."
7,0.0,NaN,0.0,NaN,1.0,NaN,NaN,1.0,0.0,1.0,0.0,1.0,TOX25232,O=C(O)Cc1cc(I)c(Oc2ccc(O)c(I)c2)c(I)c1,"[(x, [tensor([0., 0., 1., 0., 0., 0., 0., 0., ..."
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,TOX22514,OC[C@H](O)[C@@H](O)[C@H](O)CO,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX22517,CCCCCCCC(=O)[O-].CCCCCCCC(=O)[O-].[Zn+2],"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."


mol: Data(x=[16, 177], edge_index=[2, 50], edge_attr=[50, 17], num_nodes=16)


In [8]:
explore_df(tox_pyg_df)

Shape: 7,823 rows x 15 columns

Info:
<class 'pandas.DataFrame'>
RangeIndex: 7823 entries, 0 to 7822
Data columns (total 15 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   NR-AR          7258 non-null   float64
 1   NR-AR-LBD      6751 non-null   float64
 2   NR-AhR         6542 non-null   float64
 3   NR-Aromatase   5815 non-null   float64
 4   NR-ER          6186 non-null   float64
 5   NR-ER-LBD      6948 non-null   float64
 6   NR-PPAR-gamma  6443 non-null   float64
 7   SR-ARE         5825 non-null   float64
 8   SR-ATAD5       7065 non-null   float64
 9   SR-HSE         6460 non-null   float64
 10  SR-MMP         5804 non-null   float64
 11  SR-p53         6767 non-null   float64
 12  mol_id         7823 non-null   str    
 13  smiles         7823 non-null   str    
 14  mol            7823 non-null   object 
dtypes: float64(12), object(1), str(2)
memory usage: 916.9+ KB

Describe:


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles,mol
count,7258.000000,6751.000000,6542.000000,5815.000000,6186.000000,6948.000000,6443.000000,5825.000000,7065.000000,6460.000000,5804.000000,6767.000000,7823,7823,7823
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7823,7823,7823
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,1
mean,0.042436,0.035106,0.117395,0.051591,0.127869,0.050230,0.028869,0.161717,0.037367,0.057585,0.158167,0.062509,NaN,NaN,NaN
std,0.201595,0.184061,0.321915,0.221218,0.333971,0.218436,0.167450,0.368222,0.189674,0.232975,0.364929,0.242096,NaN,NaN,NaN
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN



Sample:


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles,mol
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."


### Side-by-side example

Visual check of the first molecule in both representations before running the full comparison.

In [9]:
dgl_graph = tox_dgl_df["mol"].iloc[0]
pyg_graph = tox_pyg_df["mol"].iloc[0]

print(f"SMILES: {tox_dgl_df['smiles'].iloc[0]}\n")
print(f"DGL: {dgl_graph}\n")
print(f"PyG: {pyg_graph}\n")

src, dst = dgl_graph.edges()
print(f"DGL edges:\n{torch.stack([src, dst])}\n")
print(f"PyG edges:\n{pyg_graph.edge_index}\n")

print(f"Node features equal: {torch.equal(dgl_graph.ndata['feats'], pyg_graph.x)}")
print(f"Edge features equal: {torch.equal(dgl_graph.edata['edge_feats'], pyg_graph.edge_attr)}")

SMILES: CCOc1ccc2nc(S(N)(=O)=O)sc2c1

DGL: Graph(num_nodes=16, num_edges=50,
      ndata_schemes={'feats': Scheme(shape=(177,), dtype=torch.float32)}
      edata_schemes={'edge_feats': Scheme(shape=(17,), dtype=torch.float32)})

PyG: Data(x=[16, 177], edge_index=[2, 50], edge_attr=[50, 17], num_nodes=16)

DGL edges:
tensor([[ 0, 10, 10, 11, 11, 12, 12,  4,  4,  5,  5, 15, 15,  1,  1,  7,  7,  2,
          2, 13,  2,  3,  2,  8,  7,  6,  6, 14, 14,  9,  9, 12, 14, 15,  0,  1,
          2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15],
        [10,  0, 11, 10, 12, 11,  4, 12,  5,  4, 15,  5,  1, 15,  7,  1,  2,  7,
         13,  2,  3,  2,  8,  2,  6,  7, 14,  6,  9, 14, 12,  9, 15, 14,  0,  1,
          2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15]],
       dtype=torch.int32)

PyG edges:
tensor([[ 0, 10, 10, 11, 11, 12, 12,  4,  4,  5,  5, 15, 15,  1,  1,  7,  7,  2,
          2, 13,  2,  3,  2,  8,  7,  6,  6, 14, 14,  9,  9, 12, 14, 15,  0,  1,
          2,  3,  4,  5, 

### Comparison

In [10]:
tox_summary = compare_datasets(dgl_df=tox_dgl_df, pyg_df=tox_pyg_df)
tox_refeaturised = refeaturise_sample(df=tox_dgl_df, n=REFEATURISE_SAMPLE_SIZE, seed=SEED)
results[tox21_source.value] = {"summary": tox_summary, "refeaturised": tox_refeaturised}

report(name=tox21_source.value, summary=tox_summary, refeaturised=tox_refeaturised)

=== tox21 ===
DGL shape: (7823, 15) | PyG shape: (7823, 15)
Same columns: True
Same non-mol columns (ids, smiles, labels): True

Stored graphs: 7,772 of 7,823 identical
  bondless: 31
  rdkit_drift: 20
  unexplained: 0

Re-featurised sample (current PyG code vs original dgllife code): 995 of 1,000 identical
  bondless: 5
  rdkit_drift: 0
  unexplained: 0

Stored graph differences (first 10 per category):


,index,smiles,failed_checks,category
0,95,[I-].[K+],[edge_feats],bondless
1,149,[Cu]I,[edge_feats],bondless
2,255,[Hg+2],[edge_feats],bondless
3,300,O=[Cr](=O)([O-])O[Cr](=O)(=O)[O-],[edge_feats],bondless
4,380,O=[Zr](Cl)Cl,[edge_feats],bondless
5,659,[Ba+2],[edge_feats],bondless
6,782,C/C=C(C(=C/C)/c1ccc(O)cc1)\c1ccc(O)cc1,[node_feats],rdkit_drift
7,952,Cl[Pd]Cl,[edge_feats],bondless
8,985,[TlH2+],[edge_feats],bondless
9,1397,O=[Mo](=O)=O,[edge_feats],bondless


In [11]:
# free up memory before loading the next dataset
del tox_dgl_df, tox_pyg_df, dgl_graph, pyg_graph
gc.collect()

495

## MUV

Compare molecules created for the MUV dataset.

The processed MUV and DUD-E GPCR datasets are large (~2.3GB and ~2.8GB per pickle), so only the comparison is shown and each dataset is deleted from memory afterwards.

In [12]:
muv_dgl_df = handler.load_processed_dataset(dataset_source=muv_source, feature_type=dgl_with_bonds)
muv_pyg_df = handler.load_processed_dataset(dataset_source=muv_source, feature_type=pyg_with_bonds)

show_df(df=muv_pyg_df, n=5)

,MUV-466,MUV-548,MUV-600,MUV-644,MUV-652,MUV-689,MUV-692,MUV-712,MUV-713,MUV-733,MUV-737,MUV-810,MUV-832,MUV-846,MUV-852,MUV-858,MUV-859,mol_id,smiles,mol
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,CID2999678,Cc1cccc(N2CCN(C(=O)C34CC5CC(CC(C5)C3)C4)CC2)c1C,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
1,0.0,0.0,NaN,NaN,0.0,0.0,0.0,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,0.0,0.0,CID2999679,Cn1ccnc1SCC(=O)Nc1ccc(Oc2ccccc2)cc1,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
2,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,CID2999672,COc1cc2c(cc1NC(=O)CN1C(=O)NC3(CCc4ccccc43)C1=O...,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
3,NaN,0.0,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,CID5390002,O=C1/C(=C/NC2CCS(=O)(=O)C2)c2ccccc2C(=O)N1c1cc...,"[(x, [tensor([0., 0., 1., 0., 0., 0., 0., 0., ..."
4,0.0,NaN,NaN,NaN,0.0,NaN,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CID2999670,NC(=O)NC(Cc1ccccc1)C(=O)O,"[(x, [tensor([0., 1., 0., 0., 0., 0., 0., 0., ..."


In [13]:
muv_summary = compare_datasets(dgl_df=muv_dgl_df, pyg_df=muv_pyg_df)
muv_refeaturised = refeaturise_sample(df=muv_dgl_df, n=REFEATURISE_SAMPLE_SIZE, seed=SEED)
results[muv_source.value] = {"summary": muv_summary, "refeaturised": muv_refeaturised}

report(name=muv_source.value, summary=muv_summary, refeaturised=muv_refeaturised)

=== muv ===
DGL shape: (93087, 20) | PyG shape: (93087, 20)
Same columns: True
Same non-mol columns (ids, smiles, labels): True

Stored graphs: 92,900 of 93,087 identical
  bondless: 0
  rdkit_drift: 187
  unexplained: 0

Re-featurised sample (current PyG code vs original dgllife code): 1,000 of 1,000 identical
  bondless: 0
  rdkit_drift: 0
  unexplained: 0

Stored graph differences (first 10 per category):


,index,smiles,failed_checks,category
0,296,Cc1ccc(S(=O)(=O)NC2=NC3=NC(=O)CC(c4ccc(Cl)cc4)...,"[edge_index, node_feats, edge_feats]",rdkit_drift
1,482,CCOc1ccc(C2CC(c3cccc(C)c3)N3NC(N)=NC3=N2)cc1,"[edge_index, node_feats, edge_feats]",rdkit_drift
2,526,C(=C/c1ccccc1)\C=N\Nc1nc2ccccc2s1,[edge_feats],rdkit_drift
3,1857,CS(=O)(=O)NC1=NC2=NC(c3ccc(Br)cc3)=CC(c3ccccc3...,"[edge_index, node_feats, edge_feats]",rdkit_drift
4,2159,COc1ccc(/C=N/NC(=O)c2ccc(CSc3nc(C)cc(C)n3)cc2)...,[node_feats],rdkit_drift
5,2911,CCOC1=C/C(=C2/C=C(c3ccc(OC)cc3)NC(N)=C2C#N)C=C...,"[edge_index, node_feats, edge_feats]",rdkit_drift
6,3145,CCOC1=CC(=O)/C(=C2\N=C(N)NC(C)=C2c2ccc(OC)c(OC...,"[edge_index, node_feats, edge_feats]",rdkit_drift
7,4724,COc1ccc(C2CC(c3ccc(OC)cc3)N3NC(NC(=O)c4cccs4)=...,"[edge_index, node_feats, edge_feats]",rdkit_drift
8,4898,Oc1cccc2ccc(/C=C/c3cccs3)nc12,[edge_feats],rdkit_drift
9,5548,COCC(=O)NC1=NC2=NC(c3ccc(OC)cc3)CC(c3ccccc3)N2N1,"[edge_index, node_feats, edge_feats]",rdkit_drift


In [14]:
del muv_dgl_df, muv_pyg_df
gc.collect()

1899

## DUD-E GPCR

Compare molecules created for the DUD-E GPCR dataset.

In [15]:
dude_dgl_df = handler.load_processed_dataset(dataset_source=dude_source, feature_type=dgl_with_bonds)
dude_pyg_df = handler.load_processed_dataset(dataset_source=dude_source, feature_type=pyg_with_bonds)

show_df(df=dude_pyg_df, n=5)

,smiles,adrb1,adrb2,aa2ar,drd3,cxcr4,mol
0,C#CC(=O)NC1CC(C1)[NH3+],NaN,NaN,NaN,0.0,NaN,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
1,C#CCN(C1CCCCC1)C(=O)C[C@H]2C(=O)NCC[NH+]2Cc3cc...,NaN,NaN,NaN,0.0,NaN,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
2,C#CCN(C1CCCCC1)C(=O)Nc2cccc(c2)N3CCNC3=O,NaN,NaN,0.0,NaN,NaN,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
3,C#CCN(Cc1cc(cc2c1OCOC2)[N+](=O)[O-])[C@@H]3CCc...,NaN,NaN,0.0,NaN,NaN,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."
4,C#CCN(Cc1ccco1)C(=O)C(=O)Nc2ccc(cc2)Oc3ccccc3C#N,NaN,NaN,NaN,0.0,NaN,"[(x, [tensor([1., 0., 0., 0., 0., 0., 0., 0., ..."


In [16]:
dude_summary = compare_datasets(dgl_df=dude_dgl_df, pyg_df=dude_pyg_df)
dude_refeaturised = refeaturise_sample(df=dude_dgl_df, n=REFEATURISE_SAMPLE_SIZE, seed=SEED)
results[dude_source.value] = {"summary": dude_summary, "refeaturised": dude_refeaturised}

report(name=dude_source.value, summary=dude_summary, refeaturised=dude_refeaturised)

=== dude_gpcr ===
DGL shape: (99782, 7) | PyG shape: (99782, 7)
Same columns: True
Same non-mol columns (ids, smiles, labels): True

Stored graphs: 99,483 of 99,782 identical
  bondless: 0
  rdkit_drift: 299
  unexplained: 0

Re-featurised sample (current PyG code vs original dgllife code): 1,000 of 1,000 identical
  bondless: 0
  rdkit_drift: 0
  unexplained: 0

Stored graph differences (first 10 per category):


,index,smiles,failed_checks,category
0,50,C#CCSC1=NC(=O)[C@@H]2[C@@H](CC(=O)NC2=N1)c3ccc...,[node_feats],rdkit_drift
1,286,C/C(=N\N=C/1\NC(=O)CS1)/c2ccc3ccccc3c2O,"[edge_index, node_feats, edge_feats]",rdkit_drift
2,293,C/C(=N\NC(=O)CCC(=O)N/N=C(\C)/c1ccc(cc1O)O)/c2...,[node_feats],rdkit_drift
3,499,C/C=C(\C)/c1cc(c(c2c1Oc3cc(c(c(c3C(=O)O2)C)Cl)...,"[node_feats, edge_feats]",rdkit_drift
4,563,C1/C(=N\N=C\2/NC(=O)[C@@H](S2)CC(=O)NC(=O)N)/N...,"[edge_index, node_feats, edge_feats]",rdkit_drift
5,564,C1/C(=N\N=C\2/NC(=O)[C@H](S2)CC(=O)NC(=O)N)/NC...,"[edge_index, node_feats, edge_feats]",rdkit_drift
6,576,C1C(=O)N/C(=N\N=C\2/NC(=O)[C@H](S2)CC(=O)NC(=O...,"[edge_index, node_feats, edge_feats]",rdkit_drift
7,1249,C1CS(=O)(=O)C[C@@H]1NC(=O)NC(=O)CSC2=NC(=O)CC(...,[node_feats],rdkit_drift
8,1643,C=CCNC(=S)NNC1=NC(=O)[C@H]2C(=[NH+]1)c3ccccc3C...,"[edge_index, node_feats, edge_feats]",rdkit_drift
9,1644,C=CCNC(=S)NNC1=NC(=O)[C@H]2C(=[NH+]1)c3ccccc3C...,"[edge_index, node_feats, edge_feats]",rdkit_drift


In [17]:
del dude_dgl_df, dude_pyg_df
gc.collect()

405

## Conclusion

Overall summary across all dataset sources. The final cell raises an `AssertionError` if any **unexplained** difference was found, so that a full "Run All" clearly fails if the PyG graphs ever diverge from the DGL graphs (e.g. after a package upgrade or a change to `PyGBondsDatasetCreator`).

In [18]:
conclusion_df = pd.DataFrame([
    {
        "dataset": name,
        "molecules": r["summary"]["pyg_shape"][0],
        "same_rows_and_labels": r["summary"]["same_non_mol_columns"],
        "identical_graphs": r["summary"]["pyg_shape"][0] - len(r["summary"]["differences"]),
        **{category: (r["summary"]["differences"]["category"] == category).sum() for category in CATEGORIES},
        "refeaturised_checked": r["refeaturised"]["checked"],
        "refeaturised_unexplained": (r["refeaturised"]["differences"]["category"] == "unexplained").sum(),
        "equivalent": is_equivalent(r["summary"], r["refeaturised"]),
    }
    for name, r in results.items()
])
show_df(df=conclusion_df)

,dataset,molecules,same_rows_and_labels,identical_graphs,bondless,rdkit_drift,unexplained,refeaturised_checked,refeaturised_unexplained,equivalent
0,tox21,7823,True,7772,31,20,0,1000,0,True
1,muv,93087,True,92900,0,187,0,1000,0,True
2,dude_gpcr,99782,True,99483,0,299,0,1000,0,True


In [19]:
assert conclusion_df["equivalent"].all(), "PyG graphs have unexplained differences from DGL graphs, see the reports above"
print("pyg_with_bonds graphs are equivalent to the dgl_with_bonds graphs (no unexplained differences)")

pyg_with_bonds graphs are equivalent to the dgl_with_bonds graphs (no unexplained differences)


### Notes for replicating experiments

- **`bondless` molecules:** the deprecated `DGLBondsTrainTestStrategy` dropped molecules whose `edge_feats.shape[-1] < 17`, i.e. molecules without any bonds. To train on the same molecules, the PyG train/test strategy needs to drop graphs with no bond edges, i.e. `data.num_edges == data.num_nodes` (only self-loops).
- **`rdkit_drift` molecules:** the PyG code reproduces the original dgllife code exactly in the current environment, so these differences come from the (previously unpinned) RDKit / ChEMBL structure pipeline versions used to create the DGL datasets. They only affect a handful of molecules and the specific features listed in `failed_checks`.